# KAT-VAD v2 — **E3: 2 × 2 factorial × 5 seeds** (A0–A3, seeds 2024–2028)

Pre-registration: proposal **§10.1–§10.3**, `core/docs/v2/PREREG_ADDENDUM.md` **§17 Q6** (guardrail = O1′),
**§15 G6–G7** (position reads), **§18 Amendment 9, M1–M10** (how E3 is computed). Needs the pilot on Drive: v2 dataset
dir, KNN cache, the baked caches of A1/A2/A3 (T2 + DoTA), and the four s2099 configs.

| step | what | GPU? |
|---|---|---|
| 0 | setup, deps; **Amendment 9 must be signed** (the notebook refuses a draft) | — |
| 1 | preflight; stage the v2 dir, KNN, inputs to VM disk | no |
| 2 | **train** the 20 runs (O7 recipe, only `train.seed` differs from the pilot twin), chunked + resumable, J10 | **yes** |
| 3 | **diagnostics** per arm × seed (M2): T2-val guardrails, O1′ vs A0 of the same seed, O5 on `dota_cap_dev` | yes |
| 4 | **protocol B** per arm (M3): A0/A1 on `dota_dev`, A0–A3 on `dota_cap_dev` | yes |
| 5 | **`p_T2`** position prior (G6 a) on both splits | no |
| 6 | **`e3_readout`** — the decision (M4–M8) → stop | no |

**M10:** step 6 refuses to run until all 20 runs are finished and verified. Nothing reads T2-test, DoTA-eval or
DoTA-CAP-eval (M9). A crashed run is re-run **with the same seed** (M1) — re-run the cell. The 20 runs do not fit one
Colab session: steps 2–4 skip whatever is already verified on Drive, so re-running the notebook resumes.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/, cache/ (CLIP, VideoMAE), v1 outputs
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # write: kat-vad/ (code), data/, cache/, outputs/
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V2_E3_SEEDS'), 'this upload predates Amendment 9 -- re-upload branch v2'

DATASET    = constants.DADA_ORIGIN_DATASET               # 'DADA2000_orig'
KERNEL     = constants.DADA_ORIGIN_SCORE_HEAD_KERNEL     # 3
TOPK_PCT   = constants.DADA_ORIGIN_MIL_TOPK_PCT          # 5
NUM_EPOCHS, CHUNK_EPOCHS = 20, 5                         # O7 recipe
PILOT_SEED = 2099
SEEDS      = list(constants.V2_E3_SEEDS)                 # M1: 2024-2028, never replaced or added
ARMS       = list(constants.V2_E3_ARMS)
CRN        = 'R2'                                        # D11
ENC        = constants.VIDEOMAE_ENCODER_S                # E2(d) pick (§15 G1)
MOTION_DIM = constants.VIDEOMAE_ARCH[ENC][0]             # 384

CLIP_DIR  = constants.CLIP_CACHE_DIR / DATASET
DOTA      = constants.DATA_ROOT / 'DoTA'
LABELS    = DOTA / 'labels_s8'                            # defs.json (class names)
DOTA_S1   = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
V2_DATA   = Path(DRIVE_V2) / 'data' / f'{DATASET}_v2'
V2_CACHE  = Path(DRIVE_V2) / 'cache'
KNN_V2    = V2_CACHE / 'knn' / f'{DATASET}_v2' / constants.KNN_CACHE_FILENAME
TAG       = {'A1': f'A1_{CRN}', 'A2': 'A2_S', 'A3': f'A3_{CRN}_S'}
BAKED_T2  = {arm: V2_CACHE / 'v2' / TAG[arm] / DATASET for arm in TAG}
A1_DOTA   = V2_CACHE / 'v2' / TAG['A1'] / 'DoTA_s3_from_s1'          # dota_dev bake (covers dota_cap_dev)
CAP_BAKED = {arm: V2_CACHE / 'v2' / TAG[arm] / 'DoTA_CAP_s3_from_s1' for arm in ('A2', 'A3')}
PILOT     = constants.OUTPUT_ROOT / 'v2_pilot'           # <arm>/s2099/config.yaml (the twins, M1)
E3        = constants.OUTPUT_ROOT / 'v2_e3'              # <arm>/s<seed>/{ckpt, metrics, config, diag/}; REPORTS/
LOCAL = Path('/content/e3')
LOCAL.mkdir(parents=True, exist_ok=True)

# Optional: restrict this session's training to a subset (the readout still needs all 20).
TRAIN_ARMS, TRAIN_SEEDS = ARMS, SEEDS
for name in ('V2_DATA', 'KNN_V2', 'PILOT', 'E3'):
    print(f'{name:8s}', globals()[name])

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu
python -c "import transformers, av, einops, faiss; print('transformers', transformers.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- training will be unusably slow'
df -h /content | tail -1

In [ ]:
import json
import shutil
import subprocess
import time

import numpy as np
import torch
import yaml

ENV = dict(os.environ)


def run(cmd, log=None, quiet_steps=False):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                if not (quiet_steps and 'global_step' in line):
                    print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    """Copy a Drive dir (or only ``ids`` + its manifests) / file to VM disk once (bit-identical; C2)."""
    if dst.exists() and ids is None:
        return dst
    t0 = time.time()
    if src.is_dir() and ids is None:
        shutil.copytree(src, dst)
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for m in src.glob('*.json'):
            shutil.copy2(m, dst / m.name)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """Copy a VM-local result to Drive and fsync every file (C10)."""
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        files = [p for p in dst.rglob('*') if p.is_file()]
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        files = [dst]
    for p in files:
        with open(p, 'rb+') as fh:
            os.fsync(fh.fileno())


def ckpt_step(path):
    if not path.exists():
        return 0
    return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None

## 1. Preflight + stage

In [ ]:
import yaml

from core.data.v2_splits import load_split
from core.tools import e3_readout, position_prior, protocol_b_eval, v2_dataset, v2_diagnostics

addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert '## 18. Amendment 9' in addendum, 'upload predates §18'
assert 'Not in force until the user authorizes it' not in addendum, \
    'Amendment 9 is still a draft: sign it in PREREG_ADDENDUM.md §18 (and commit) before the first E3 run'
dev = load_split(constants.V2_SPLIT_DOTA_DEV)
cap_dev = load_split(constants.V2_SPLIT_DOTA_CAP_DEV)
assert len(cap_dev) == 569 and set(cap_dev) <= set(dev), (len(cap_dev), len(dev))
for path in (V2_DATA / v2_dataset.MANIFEST_FILENAME, KNN_V2, A1_DOTA, LABELS / constants.DEFS_FILENAME,
             *BAKED_T2.values(), *CAP_BAKED.values()):
    assert path.exists(), f'missing {path} (published by the pilot)'
a1 = json.loads((BAKED_T2['A1'] / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
for arm, d in BAKED_T2.items():           # M1: one train split for every arm
    m = json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
    assert m['train_ids_sha1'] == a1['train_ids_sha1'], f'{arm} baked on another train split'
TWIN = {arm: yaml.safe_load((PILOT / arm / f's{PILOT_SEED}' / 'config.yaml').read_text()) for arm in ARMS}

L_V2 = stage(V2_DATA, LOCAL / 'stage' / 'corpus_v2')
L_KNN = stage(KNN_V2, LOCAL / 'stage' / 'knn_v2.npz')
L_CLIP = stage(CLIP_DIR, LOCAL / 'stage' / 'clip')
L_T2 = {'A0': L_CLIP, **{arm: stage(d, LOCAL / 'stage' / f'{arm}_T2') for arm, d in BAKED_T2.items()}}
L_DOTA_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'DoTA_s1_ncc_dev', dev)
L_A1_DOTA = stage(A1_DOTA, LOCAL / 'stage' / 'A1_DOTA')
L_CAP = {arm: stage(d, LOCAL / 'stage' / f'{arm}_CAP') for arm, d in CAP_BAKED.items()}
L_DOTA_LABELS = stage(LABELS / constants.DEFS_FILENAME,
                      LOCAL / 'stage' / 'dota_labels' / constants.DEFS_FILENAME).parent
print(f'DoTA-dev {len(dev)} clips, DoTA-CAP-dev {len(cap_dev)}; every arm on train split {a1["train_ids_sha1"][:12]}')

## 2. Train the 20 runs — chunked, resumable, verified (M1)

O7's recipe; each run's config must differ from its s2099 twin **only** in `train.seed`. Skip/resume reads the
**checkpoint's own `global_step`** (J10). Each finished run prints its wall time; the ETA is from the runs done in this
session. **Re-run this cell after a disconnect.**

In [ ]:
MOTION = ['--set', f'v2.motion={ENC}', '--set', f'model.motion_dim={MOTION_DIM}']
EXTRA = {'A0': [], 'A1': ['--set', f'v2.crn={CRN}'], 'A2': MOTION, 'A3': ['--set', f'v2.crn={CRN}', *MOTION]}


def run_dir(arm, seed):
    return E3 / arm / f's{seed}'


def train_cmd(arm, seed, out_dir):
    return [sys.executable, '-m', 'core.train',
            '--set', 'train.stage=2', '--set', f'train.seed={seed}',
            '--set', f'train.num_epochs={NUM_EPOCHS}', '--set', 'train.amp=true',
            '--set', f'data.dataset={DATASET}',
            '--set', f'model.score_head_kernel={KERNEL}', '--set', f'loss.mil_topk_pct={TOPK_PCT}',
            '--set', 'kip.enabled=false', *EXTRA[arm],
            '--data-dir', L_V2, '--clip-dir', L_T2[arm], '--knn-cache', L_KNN,
            '--output-dir', out_dir]


def sync(local, drive_dir):
    drive_dir.mkdir(parents=True, exist_ok=True)
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        if (local / name).exists():
            publish(local / name, drive_dir / name)


def verified(d):
    last = metrics_last(d / 'metrics.jsonl')
    return bool(last and last['epoch'] + 1 == NUM_EPOCHS
                and ckpt_step(d / 'checkpoint_last.pt') == last['global_step'])


def config_diff(a, b):
    out = set()
    for section in set(a) | set(b):
        sa, sb = a.get(section) or {}, b.get(section) or {}
        out |= {f'{section}.{k}' for k in set(sa) | set(sb) if sa.get(k) != sb.get(k)}
    return out


durations = []
todo = [(a, s) for s in TRAIN_SEEDS for a in TRAIN_ARMS if not verified(run_dir(a, s))]
print(f'{len(todo)} runs to train in this session')
for k, (arm, seed) in enumerate(todo):
    drive_dir, local = run_dir(arm, seed), LOCAL / 'runs' / arm / f's{seed}'
    local.mkdir(parents=True, exist_ok=True)
    last = metrics_last(drive_dir / 'metrics.jsonl')
    if not (local / 'checkpoint_last.pt').exists() and last and \
            ckpt_step(drive_dir / 'checkpoint_last.pt') == last['global_step']:
        for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
            shutil.copy2(drive_dir / name, local / name)          # resume from a consistent Drive copy
    t0 = time.time()
    while True:
        done = (metrics_last(local / 'metrics.jsonl') or {'epoch': -1})['epoch'] + 1
        if done >= NUM_EPOCHS:
            break
        cmd = train_cmd(arm, seed, local)
        if done:
            cmd += ['--resume', local / 'checkpoint_last.pt']
        cmd += ['--stop-after-epochs', min(CHUNK_EPOCHS, NUM_EPOCHS - done)]
        print(f'=== {arm} s{seed}: epochs {done} -> {min(done + CHUNK_EPOCHS, NUM_EPOCHS)} of {NUM_EPOCHS}')
        run(cmd, log=LOCAL / f'{arm}_s{seed}_train.log', quiet_steps=True)
        sync(local, drive_dir)
    sync(local, drive_dir)
    assert verified(drive_dir), f'{arm} s{seed}: J10 FAILS on Drive'
    diff = config_diff(TWIN[arm], yaml.safe_load((drive_dir / 'config.yaml').read_text()))
    assert diff == {'train.seed'}, f'{arm} s{seed} differs from its s{PILOT_SEED} twin beyond the seed: {diff}'
    durations.append(time.time() - t0)
    eta = sum(durations) / len(durations) * (len(todo) - k - 1)
    print(f'{arm} s{seed}: step {metrics_last(drive_dir / "metrics.jsonl")["global_step"]}, '
          f'{durations[-1] / 60:.1f} min; ETA this session {eta / 3600:.1f} h')
missing = [(a, s) for s in SEEDS for a in ARMS if not verified(run_dir(a, s))]
print('all 20 runs verified' if not missing else f'still to train: {missing}')

## 3. Diagnostics per arm × seed (M2)

A0 of each seed first: A1–A3 read **A0's diag of the same seed** for O1′. O5 on `dota_cap_dev` (unlabelled).

In [ ]:
DOTA_IN = {'A0': L_DOTA_S1, 'A1': L_A1_DOTA, **L_CAP}


def diagnose(arm, seed):
    d = run_dir(arm, seed)
    out = d / e3_readout.DIAG_DIR
    if (out / v2_diagnostics.DIAG_JSON).exists():
        return
    assert verified(d), f'{arm} s{seed} not trained'
    a0 = run_dir('A0', seed) / e3_readout.DIAG_DIR / v2_diagnostics.DIAG_JSON
    run([sys.executable, '-m', 'core.tools.v2_diagnostics', '--run', f'{arm}_s{seed}', d / 'checkpoint_last.pt',
         '--data-dir', L_V2, '--t2-input-dir', L_T2[arm], '--dota-input-dir', DOTA_IN[arm],
         '--dota-s1-dir', L_DOTA_S1, '--dota-data-dir', L_DOTA_LABELS, '--dota-split', constants.V2_SPLIT_DOTA_CAP_DEV,
         *(['--a0-diag', a0] if arm != 'A0' else []), '--out-dir', out, '--device', 'auto'],
        log=LOCAL / f'{arm}_s{seed}_diag.log')


for seed in SEEDS:
    for arm in ARMS:                       # A0 first: the others need its diag
        if verified(run_dir(arm, seed)):
            diagnose(arm, seed)
print('diagnosed:', sum((run_dir(a, s) / e3_readout.DIAG_DIR / v2_diagnostics.DIAG_JSON).exists()
                        for a in ARMS for s in SEEDS), 'of 20')

## 4. Protocol B per arm (M3)

All 5 checkpoints of an arm in one call (`--run s<seed>`), seed-averaged per frame; `per_run_macro` gives the per-seed
macro that the decision interval pairs. A0 and A1 on `dota_dev` **and** `dota_cap_dev`; A2/A3 on `dota_cap_dev` only.

In [ ]:
SPLIT_INPUT = {
    ('A0', constants.V2_SPLIT_DOTA_DEV): L_DOTA_S1, ('A0', constants.V2_SPLIT_DOTA_CAP_DEV): L_DOTA_S1,
    ('A1', constants.V2_SPLIT_DOTA_DEV): L_A1_DOTA, ('A1', constants.V2_SPLIT_DOTA_CAP_DEV): L_A1_DOTA,
    ('A2', constants.V2_SPLIT_DOTA_CAP_DEV): L_CAP['A2'], ('A3', constants.V2_SPLIT_DOTA_CAP_DEV): L_CAP['A3'],
}
missing = [(a, s) for s in SEEDS for a in ARMS if not verified(run_dir(a, s))]
assert not missing, f'M10: train every run first: {missing}'
for (arm, split), input_dir in SPLIT_INPUT.items():
    out = e3_readout.pb_dir(E3, arm, split)
    if (out / protocol_b_eval.CLIP_SCORES_NPZ).exists():
        print(f'{arm}/{split}: reusing')
        continue
    runs = [x for s in SEEDS for x in ('--run', f's{s}', run_dir(arm, s) / 'checkpoint_last.pt')]
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', *runs,
         '--input-dir', input_dir, '--s1-dir', L_DOTA_S1,
         '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--data-dir', L_DOTA_LABELS, '--split', split, '--out-dir', out, '--device', 'auto'],
        log=LOCAL / f'{arm}_{split}_pb.log')
print('protocol B done for', len(SPLIT_INPUT), 'arm x split pairs')

## 5. T2 position prior `p_T2` (G6 a) — no GPU

In [ ]:
for split in (constants.V2_SPLIT_DOTA_DEV, constants.V2_SPLIT_DOTA_CAP_DEV):
    out = E3 / e3_readout.REPORTS / e3_readout.PRIOR_DIR / split
    if not (out / position_prior.PRIOR_NPZ).exists():
        run([sys.executable, '-m', 'core.tools.position_prior', '--data-dir', L_V2, '--s1-dir', L_DOTA_S1,
             '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
             '--split', split, '--out-dir', out])
    print((out / position_prior.READOUT_MD).read_text())

## 6. Read-out and decision (M4–M8)

In [ ]:
missing = [(a, s) for s in SEEDS for a in ARMS
           if not (run_dir(a, s) / e3_readout.DIAG_DIR / v2_diagnostics.DIAG_JSON).exists()]
assert not missing, f'M10: diagnose every run first: {missing}'
run([sys.executable, '-m', 'core.tools.e3_readout', '--e3-dir', E3])
print((E3 / e3_readout.REPORTS / e3_readout.E3_READOUT_MD).read_text())

## 7. Record — then **stop**

Paste `v2_e3/REPORTS/e3_readout.md` back to Claude (and `e3_readout.json` if asked). The decision is mechanical
(§10.3, M4–M5); nothing is re-run, re-seeded or re-tuned after reading it. The Final step (T2-test, DoTA-eval,
DoTA-CAP-eval, opened once) is a separate notebook, written after the adoption is recorded (M9).